# Phase 2: Data Preprocessing & Train/Test Pipeline

**Project:** AI-Based Predictive Maintenance System for CNC Machines in Automotive Manufacturing Using Industrial Sensor Data  
**Objective:** Build a robust, leakage-free preprocessing pipeline, reproducible stratified train/test split, and training-only SMOTE oversampling.

---

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# Setup paths
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from config import (
    RAW_DATA_PATH, PROCESSED_DATA_DIR, MODELS_DIR, METRICS_DIR, TABLES_DIR,
    RANDOM_STATE, ID_COLS, TARGET_COL, FAILURE_TYPE_COLS, FEATURE_COLS, PREPROCESSOR_PATH
)
from src.data_loader import load_raw_data, verify_dataset_integrity
from src.preprocessing import (
    separate_features_and_target,
    split_data,
    create_preprocessor,
    fit_and_transform_features,
    apply_smote,
    transform_single_instance,
    execute_pipeline
)

print("Modules imported successfully.")

## 2. Load Raw Dataset and Data Quality Verification

In [ ]:
df_raw = load_raw_data(RAW_DATA_PATH)
integrity = verify_dataset_integrity(df_raw)
print(f"Raw dataset shape: {df_raw.shape}")
print(f"Total missing values: {integrity['total_nulls']}")
print(f"Duplicate rows: {integrity['duplicate_count']}")
print(f"Dataset is valid: {integrity['is_valid']}")
df_raw.head()

## 3. Feature and Target Separation (Data Leakage Prevention)

- **Target:** `Machine failure`
- **Excluded Identifiers:** `UDI`, `Product ID` (non-predictive metadata)
- **Excluded Failure Mode Flags:** `TWF`, `HDF`, `PWF`, `OSF`, `RNF` (direct failure indicators that cause target leakage)
- **Operating Features:** `Type`, `Air temperature [K]`, `Process temperature [K]`, `Rotational speed [rpm]`, `Torque [Nm]`, `Tool wear [min]`

In [ ]:
X, y = separate_features_and_target(df_raw)
print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape:   {y.shape}")
print(f"Selected input features: {list(X.columns)}")

## 4. Stratified Train / Test Split (80% / 20%)

Splitting is strictly executed **before** fitting the preprocessor or applying SMOTE.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = split_data(X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f"Training set samples: {len(y_train)} (Healthy: {(y_train==0).sum()}, Failures: {(y_train==1).sum()}, Failure Rate: {y_train.mean()*100:.2f}%)")
print(f"Test set samples:     {len(y_test)} (Healthy: {(y_test==0).sum()}, Failures: {(y_test==1).sum()}, Failure Rate: {y_test.mean()*100:.2f}%)")

## 5. Fit Preprocessing Pipeline on Training Data Only

- `Type`: One-Hot Encoded (`Type_H`, `Type_L`, `Type_M`).
- Numerical Features: Passed through without scaling to preserve physical engineering units.

In [ ]:
preprocessor = create_preprocessor()
X_train_proc, X_test_proc, fitted_preprocessor, feature_names = fit_and_transform_features(
    preprocessor, X_train_raw, X_test_raw
)

print(f"Transformed X_train shape: {X_train_proc.shape}")
print(f"Transformed X_test shape:  {X_test_proc.shape}")
print(f"Encoded feature names ({len(feature_names)}): {feature_names}")
X_train_proc.head()

## 6. SMOTE Oversampling on Training Data Only

SMOTE is applied strictly to `X_train_proc` and `y_train`. The test set remains untouched.

In [ ]:
X_train_smote, y_train_smote = apply_smote(X_train_proc, y_train, random_state=RANDOM_STATE)

print(f"Post-SMOTE X_train shape: {X_train_smote.shape}")
print(f"Post-SMOTE class counts:\n{y_train_smote.value_counts()}")
print(f"Test set remains untouched with shape: {X_test_proc.shape} and class counts:\n{y_test.value_counts()}")

## 7. Preprocessor Single-Machine Inference Consistency Test

In [ ]:
sample_input = {
    "Type": "M",
    "Air temperature [K]": 298.2,
    "Process temperature [K]": 308.7,
    "Rotational speed [rpm]": 1408.0,
    "Torque [Nm]": 46.3,
    "Tool wear [min]": 3.0
}

transformed_sample = transform_single_instance(sample_input, fitted_preprocessor)
print("Transformed single machine input:")
transformed_sample

## 8. Execute Pipeline and Verify Artifacts

In [ ]:
summary = execute_pipeline()
summary_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "preprocessing_summary.csv")
summary_df